## Importing Packages

In [1]:
import pandas as pd
import numpy as np
import polars as pl
import glob

import matplotlib.pyplot as plt
import seaborn as sns

from tvDatafeed import TvDatafeed, Interval
import backtesting

import os

/opt/anaconda3/lib/python3.13/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

## Creating the Dataset

### Logging into TVFeed

In [2]:
tv = TvDatafeed(username = 'jaganathapandiyan12', password = 'PASS$1234TO5678')

error while signin
you are using nologin method, data you access may be limited


### Downloading the stocks in NIFTY 500 (list downloaded from NSE website on 16th November)

In [3]:
data_nifty500 = pl.read_csv('ind_nifty500list.csv')

symbols_nifty500 = data_nifty500['Symbol'].to_list()
symbols_nifty500 = [s.replace("-", "_") for s in symbols_nifty500]
symbols_nifty500.remove('DUMMYSKFIN')


print(f'Total stocks: {len(symbols_nifty500)}')

Total stocks: 501


In [4]:
failed = []

if not os.path.exists("Data/Processed/Prices_all.parquet"):
    for symbol in symbols_nifty500:
        try:
            df = tv.get_hist(symbol = symbol, exchange = 'NSE', interval = Interval.in_daily, n_bars = 5000)
            df.to_parquet(f"data/raw/prices/{symbol}.parquet")
        
        except Exception as e:
            failed.append(symbol)


    while len(failed) != 0:
        for symbol in failed[:]:
            try:
                df = tv.get_hist(symbol = symbol, exchange = 'NSE', interval = Interval.in_daily, n_bars = 5000)
                df.to_parquet(f"data/raw/prices/{symbol}.parquet")
                failed.remove(symbol)
            
            except:
                pass

    print('\nFailed cases were run repeatedly until success. Data is complete')

else:
    print('Skipping as processed data file already exists')

Skipping as processed data file already exists


### Merging the files into a single file for analysis

In [5]:
RAW_DIR = "Data/Raw/Prices"
OUT_FILE = "Data/Processed/Prices_all.parquet"

def merge_parquet_files():
    if os.path.exists(OUT_FILE):
        print("Skipping to avoid overwrite as merged data file already exists")
        return

    files = glob.glob(os.path.join(RAW_DIR, "*.parquet"))

    if len(files) == 0:
        print("No parquet files found in data/raw/prices/")
        return

    dfs = []

    for f in files:
        symbol = os.path.basename(f).replace(".parquet", "")
        
        df = (pl.read_parquet(f).with_columns([pl.lit(symbol).alias("symbol"), 
                                               pl.col("datetime").cast(pl.Date).alias("date")
                                               ]).drop("datetime"))
        dfs.append(df)

    final_df = pl.concat(dfs, how = "vertical")

    final_df = final_df.sort(["symbol", "date"])

    final_df.write_parquet(OUT_FILE)

    print(f"Master datafile created: {OUT_FILE}")
    print(f"\nRows: {final_df.height}, Columns: {final_df.width}")


if __name__ == "__main__":
    merge_parquet_files()

Skipping to avoid overwrite as merged data file already exists


## Calculating the Required Ratios

#### Loading and sorting by stock, date to ensure that chronology is maintained

In [6]:
df = pl.read_parquet("Data/Processed/Prices_all.parquet")
df = df.sort(["symbol", "date"])

#### Left joining with industry of the NIFTY 500 stocks from NSE website

In [7]:
df = df.rename({c: c.lower().strip() for c in df.columns})
data_nifty500 = data_nifty500.rename({c: c.lower().strip() for c in data_nifty500.columns})


df = df.with_columns(
    pl.col("symbol").str.to_uppercase().alias("symbol"))

data_nifty500 = data_nifty500.with_columns(
    pl.col("symbol").str.to_uppercase().alias("symbol"))

In [8]:
df = df.join(data_nifty500, on = "symbol", how = "left")

### 1) MACD (12-day EMA - 26-day EMA)

In [9]:
df = df.with_columns([
    pl.col("close").ewm_mean(span = 12).over("symbol").alias("ema_12"),
    pl.col("close").ewm_mean(span = 26).over("symbol").alias("ema_26"),]).with_columns([
    (pl.col("ema_12") - pl.col("ema_26")).alias("macd")])

### 2) 14-day ROC

In [10]:
df = df.with_columns([
    ((pl.col("close") - pl.col("close").shift(14)) 
     / pl.col("close").shift(14)).over("symbol").alias("roc_14")])

### 3) 14-day ADX

In [11]:
df = df.with_columns([
    pl.col("high").shift(1).over("symbol").alias("prev_high"),
    pl.col("low").shift(1).over("symbol").alias("prev_low"),
    pl.col("close").shift(1).over("symbol").alias("prev_close"),])

df = df.with_columns([
    (pl.col("high") - pl.col("low")).alias("hl_range"),
    (pl.col("high") - pl.col("prev_close")).abs().alias("hc_range"),
    (pl.col("low") - pl.col("prev_close")).abs().alias("lc_range"),])

df = df.with_columns([
    pl.max_horizontal([
        pl.col("hl_range"),
        pl.col("hc_range"),
        pl.col("lc_range"),
    ]).alias("tr")])

df = df.with_columns([
    (pl.col("high") - pl.col("prev_high")).alias("up_move"),
    (pl.col("prev_low") - pl.col("low")).alias("down_move"),])

df = df.with_columns([
    pl.when(
        (pl.col("up_move") > pl.col("down_move")) & (pl.col("up_move") > 0)
        ).then(pl.col("up_move")).otherwise(0).alias("plus_dm"),

    pl.when(
        (pl.col("down_move") > pl.col("up_move")) & (pl.col("down_move") > 0)
        ).then(pl.col("down_move")).otherwise(0).alias("minus_dm"),])

df = df.with_columns([
    pl.col("tr").rolling_mean(14).over("symbol").alias("atr_14"),
    pl.col("plus_dm").rolling_mean(14).over("symbol").alias("plus_dm_14"),
    pl.col("minus_dm").rolling_mean(14).over("symbol").alias("minus_dm_14"),])


df = df.with_columns([
    (100 * pl.col("plus_dm_14") / pl.col("atr_14")).alias("plus_di_14"),
    (100 * pl.col("minus_dm_14") / pl.col("atr_14")).alias("minus_di_14"),])


df = df.with_columns([
    (100 * (pl.col("plus_di_14") - pl.col("minus_di_14")).abs()
     / (pl.col("plus_di_14") + pl.col("minus_di_14"))
    ).alias("dx_14")])

df = df.with_columns([
    pl.col("dx_14").rolling_mean(14).over("symbol").alias("adx_14")])

### 4) 5-day VWAP

In [12]:
df = df.with_columns([
    ((pl.col("high") + pl.col("low") + pl.col("close")) / 3).alias("typical_price")])

df = df.with_columns([
    (pl.col("typical_price") * pl.col("volume")).rolling_sum(5).over("symbol").alias("tp_vol_sum_5"),
    pl.col("volume").rolling_sum(5).over("symbol").alias("vol_sum_5"),
                    ]).with_columns([(pl.col("tp_vol_sum_5") / pl.col("vol_sum_5")).alias("vwap_5")])

### 5) 14-day RSI

In [13]:
df = df.with_columns(
    pl.col("close").diff().over("symbol").alias("delta"))

df = df.with_columns([
    pl.col("delta").clip(lower_bound=0).alias("gain"),
    (-pl.col("delta").clip(upper_bound=0)).alias("loss")])

df = df.with_columns([
    pl.col("gain").rolling_mean(14).over("symbol").alias("avg_gain_14"),
    pl.col("loss").rolling_mean(14).over("symbol").alias("avg_loss_14")])

df = df.with_columns((
    100 - 100 / (1 + (pl.col("avg_gain_14") / pl.col("avg_loss_14")))).alias("rsi_14"))

### 6) 20-day Volume

In [14]:
df = df.with_columns(
    pl.col("volume").rolling_mean(20).over("symbol").alias("sma_vol_20"))

### 7) 14-day ATR

In [15]:
df = df.with_columns([
    pl.col("close").shift(1).over("symbol").alias("prev_close")])

df = df.with_columns([
    pl.max_horizontal([pl.col("high") - pl.col("low"),
                       (pl.col("high") - pl.col("prev_close")).abs(),
                       (pl.col("low")  - pl.col("prev_close")).abs()]).alias("true_range")])

df = df.with_columns([
    pl.col("true_range").rolling_mean(14).over("symbol").alias("atr_14")])

### 8) Creating correlation pairs on returns

In [16]:
df = df.with_columns([
    pl.col("close").pct_change().over("symbol").alias("ret"),
    pl.col("close").pct_change().shift(1).over("symbol").alias("ret_lag1")])

In [18]:
base = df.select(["date", "symbol", "industry", "ret", "ret_lag1"])

pairs = (
    base.join(
        base, 
        on = "date", 
        how = "inner", 
        suffix = "_peer"))

In [20]:
pairs = pairs.filter(
    (pl.col("symbol") != pl.col("symbol_peer")) &
    (pl.col("industry") == pl.col("industry_peer")))

In [43]:
pairs = pairs.with_columns([
    pl.corr(
        pl.col("ret"),
        pl.col("ret_lag1_peer"))
        .over(["symbol", "symbol_peer"])
        .alias("corr")])

In [67]:
pairs_corr = (
    pairs
    .group_by(["symbol", "symbol_peer"])
    .agg([
        pl.corr(pl.col("ret"), pl.col("ret_lag1_peer")).alias("corr").filter(pl.col("corr").is_not_nan())
    ])
)

In [ ]:
top_peers = (
    pairs_corr
    .group_by("symbol")
    .agg([
        pl.struct(["symbol_peer", "corr"])
        .sort_by("corr", descending = True)
        .head(3)
        .alias("top3_peers")
    ])
)

In [72]:
top_peers

symbol,top3_peers
str,list[struct[2]]
"""AFCONS""","[{""LT"",[0.275776, 0.275776, … 0.275776]}, {""KEC"",[0.258718, 0.258718, … 0.258718]}, {""NCC"",[0.249283, 0.249283, … 0.249283]}]"
"""CROMPTON""","[{""KAJARIACER"",[0.085822, 0.085822, … 0.085822]}, {""HAVELLS"",[0.058059, 0.058059, … 0.058059]}, {""BATAINDIA"",[0.051913, 0.051913, … 0.051913]}]"
"""HINDUNILVR""","[{""HONASA"",[0.032446, 0.032446, … 0.032446]}, {""RADICO"",[0.023187, 0.023187, … 0.023187]}, {""BIKAJI"",[0.018063, 0.018063, … 0.018063]}]"
"""GICRE""","[{""IDFCFIRSTB"",[0.109949, 0.109949, … 0.109949]}, {""BAJAJHFL"",[0.10355, 0.10355, … 0.10355]}, {""INDUSINDBK"",[0.092211, 0.092211, … 0.092211]}]"
"""HONAUT""","[{""SIEMENS"",[0.101682, 0.101682, … 0.101682]}, {""INOXINDIA"",[0.094255, 0.094255, … 0.094255]}, {""ASHOKLEY"",[0.09288, 0.09288, … 0.09288]}]"
…,…
"""DCMSHRIRAM""","[{""GODREJIND"",[0.037683, 0.037683, … 0.037683]}, {""3MINDIA"",[-0.001849, -0.001849, … -0.001849]}]"
"""MMTC""","[{""GMRAIRPORT"",[0.104995, 0.104995, … 0.104995]}, {""SCI"",[0.075537, 0.075537, … 0.075537]}, {""FSL"",[0.070536, 0.070536, … 0.070536]}]"
"""BAYERCROP""","[{""UPL"",[0.068723, 0.068723, … 0.068723]}, {""PCBL"",[0.05895, 0.05895, … 0.05895]}, {""TATACHEM"",[0.058464, 0.058464, … 0.058464]}]"


In [69]:
top_peers_flat = (
    top_peers
    .explode("top3_peers")
    .select([
        pl.col("symbol"),
        pl.col("top3_peers").struct.field("symbol_peer").alias("peer"),
        pl.col("top3_peers").struct.field("corr").alias("corr")
    ])
)

In [71]:
top_peers_flat.sort("corr", descending = True).head(20)

symbol,peer,corr
str,str,list[f64]
"""ATHERENERG""","""TVSMOTOR""","[0.308578, 0.308578, … 0.308578]"
"""AFCONS""","""LT""","[0.275776, 0.275776, … 0.275776]"
"""ENRIN""","""SUZLON""","[0.27512, 0.27512, … 0.27512]"
"""ZFCVINDIA""","""ATHERENERG""","[0.262877, 0.262877, … 0.262877]"
"""AFCONS""","""KEC""","[0.258718, 0.258718, … 0.258718]"
…,…,…
"""JYOTICNC""","""CUMMINSIND""","[0.21291, 0.21291, … 0.21291]"
"""ENRIN""","""POLYCAB""","[0.21259, 0.21259, … 0.21259]"
"""SAGILITY""","""NEWGEN""","[0.210957, 0.210957, … 0.210957]"


### Dropping unnecessary columns and removing NULL rows

In [15]:
df = df.drop(["ema_12", "ema_26", "delta", "gain", "loss", "avg_gain_14", "avg_loss_14", "typical_price", 
              "tp_vol_sum_5", "vol_sum_5", "prev_close", "true_range", "prev_high", "prev_low", "prev_close", 
              "hl_range", "hc_range", "lc_range", "up_move", "down_move", "plus_dm", "minus_dm", "plus_dm_14", 
              "minus_dm_14", "dx_14", "tr", "plus_di_14", "minus_di_14"])

In [16]:
print(f'Before dropping nulls: {df.shape}')

df = df.drop_nulls()

print(f'After dropping nulls: {df.shape}')

Before dropping nulls: (1732923, 14)
After dropping nulls: (1719921, 14)


In [36]:
df.head()

symbol,open,high,low,close,volume,date,company name,industry,series,isin code
str,f64,f64,f64,f64,f64,date,str,str,str,str
"""360ONE""",302.5,317.625,302.5,317.625,7.10986e6,2019-09-19,"""360 ONE WAM Ltd.""","""Financial Services""","""EQ""","""INE466L01038"""
"""360ONE""",332.25,333.5,332.25,333.5,2.760648e6,2019-09-20,"""360 ONE WAM Ltd.""","""Financial Services""","""EQ""","""INE466L01038"""
"""360ONE""",350.175,350.175,350.175,350.175,34096.0,2019-09-23,"""360 ONE WAM Ltd.""","""Financial Services""","""EQ""","""INE466L01038"""
"""360ONE""",367.675,367.675,367.675,367.675,2.496652e6,2019-09-24,"""360 ONE WAM Ltd.""","""Financial Services""","""EQ""","""INE466L01038"""
"""360ONE""",386.05,386.05,349.3,351.2375,530580.0,2019-09-25,"""360 ONE WAM Ltd.""","""Financial Services""","""EQ""","""INE466L01038"""


## Setting up the Strategies

### Strategy 1 (Low Risk Low Reward)

### Strategy 2 (Medium Risk Medium Reward)

In [ ]:
df = df.with_columns([
    ((pl.col("macd") > 0) &
     (pl.col("roc_14") > 0) &
     (pl.col("adx_14") > 25)
     ).alias("s2_buy_signal")])

### Strategy 3 (High Risk High Reward)

In [ ]:
df = df.with_columns([
    ((pl.col("close") > pl.col("vwap_5")) &
     (pl.col("roc_14") < 60) &
     (pl.col("volume") > ("sma_vol_20"))
     ).alias("s3_buy_signal")])

## Stock selection

## Performance Assessment

## Portfolio Reallocation

## Backtesting